# Flask Deployment for AI Model

This notebook demonstrates how to deploy the trained AI model as a Flask web service. The deployment includes:

1. Setting up a Flask server with proper API endpoints
2. Implementing efficient batching and caching for improved performance
3. Adding proper error handling and logging
4. Testing the deployed model with various inputs
5. Measuring performance metrics like latency and throughput

Let's start by importing the necessary libraries.

In [ ]:
import os
import sys
import json
import time
import yaml
import logging
import requests
import threading
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from typing import Dict, List, Any, Optional
from tqdm.notebook import tqdm

# Add the project root to the path
project_root = Path().absolute().parent
sys.path.append(str(project_root))

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(name)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## 1. Load Configuration

First, let's load the configuration file that contains the model and deployment settings.

In [ ]:
# Load configuration
config_path = project_root / 'configs' / 'config.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract relevant configurations
model_config = config['model']
flask_config = config['deployment']['flask']

# Create output directory
output_dir = project_root / 'outputs' / 'flask_deployment'
os.makedirs(output_dir, exist_ok=True)

print(f"Model configuration:\n{model_config}\n")
print(f"Flask configuration:\n{flask_config}")

## 2. Prepare Model and Tokenizer Paths

Now, let's set up the paths to the trained model and tokenizer.

In [ ]:
# Define paths
model_path = project_root / 'outputs' / 'checkpoints' / 'model_final.pt'
tokenizer_path = project_root / 'outputs' / 'tokenizer' / 'tokenizer.json'
config_json_path = output_dir / 'config.json'

# Convert YAML config to JSON for the Flask server
with open(config_json_path, 'w') as f:
    json.dump(config, f, indent=2)

print(f"Model path: {model_path}")
print(f"Tokenizer path: {tokenizer_path}")
print(f"Config JSON path: {config_json_path}")

## 3. Start the Flask Server

Now, let's start the Flask server in a separate process.

In [ ]:
def start_flask_server(model_path, tokenizer_path, config_path, host='0.0.0.0', port=5000):
    """Start the Flask server in a separate process."""
    # Command to start the Flask server
    cmd = [
        sys.executable,
        str(project_root / 'src' / 'deployment' / 'flask_server.py'),
        '--model-path', str(model_path),
        '--tokenizer-path', str(tokenizer_path),
        '--config-path', str(config_path),
        '--host', host,
        '--port', str(port),
        '--batch-size', str(flask_config.get('batch_size', 8)),
        '--max-cache-size', str(flask_config.get('max_cache_size', 1000)),
        '--max-sequence-length', str(model_config.get('max_position_embeddings', 1024))
    ]
    
    # Start the server
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )
    
    # Wait for the server to start
    logger.info("Starting Flask server...")
    time.sleep(5)  # Give the server some time to start
    
    # Check if the server is running
    try:
        response = requests.get(f"http://{host}:{port}/health")
        if response.status_code == 200:
            logger.info(f"Flask server is running at http://{host}:{port}")
            return process
        else:
            logger.error(f"Flask server returned status code {response.status_code}")
            process.terminate()
            return None
    except requests.exceptions.ConnectionError:
        logger.error("Could not connect to Flask server")
        # Print server output
        stdout, stderr = process.communicate(timeout=1)
        logger.error(f"Server stdout: {stdout}")
        logger.error(f"Server stderr: {stderr}")
        process.terminate()
        return None

# Start the Flask server
host = flask_config.get('host', '0.0.0.0')
port = flask_config.get('port', 5000)

server_process = start_flask_server(
    model_path=model_path,
    tokenizer_path=tokenizer_path,
    config_path=config_json_path,
    host=host,
    port=port
)

## 4. Test the Flask API

Now that the server is running, let's test the API with some sample inputs.

In [ ]:
def generate_text(text, max_new_tokens=50, temperature=1.0, host='0.0.0.0', port=5000):
    """Generate text using the Flask API."""
    url = f"http://{host}:{port}/generate"
    payload = {
        "text": text,
        "max_new_tokens": max_new_tokens,
        "temperature": temperature
    }
    
    try:
        response = requests.post(url, json=payload)
        if response.status_code == 200:
            return response.json()
        else:
            logger.error(f"API returned status code {response.status_code}")
            logger.error(f"Response: {response.text}")
            return {"error": f"API returned status code {response.status_code}"}
    except requests.exceptions.ConnectionError:
        logger.error("Could not connect to API")
        return {"error": "Could not connect to API"}

# Test with a simple example
test_input = "def fibonacci(n):"
result = generate_text(test_input, host=host, port=port)

print(f"Input: {test_input}")
print(f"Generated text: {result.get('generated_text', 'Error: ' + result.get('error', 'Unknown error'))}")
print(f"Generation time: {result.get('generation_time', 'N/A')} seconds")
print(f"Number of tokens: {result.get('num_tokens', 'N/A')}")

## 5. Benchmark Performance

Let's benchmark the API's performance with various inputs and settings.

In [ ]:
def benchmark_api(inputs, max_new_tokens=50, temperature=1.0, host='0.0.0.0', port=5000, n_runs=3):
    """Benchmark the API with multiple inputs."""
    results = []
    
    for input_text in tqdm(inputs, desc="Benchmarking API"):
        # Run multiple times to get average performance
        run_results = []
        for _ in range(n_runs):
            start_time = time.time()
            response = generate_text(input_text, max_new_tokens, temperature, host, port)
            end_time = time.time()
            
            # Calculate total time (including network latency)
            total_time = end_time - start_time
            
            # Get generation time (server-side processing time)
            generation_time = response.get('generation_time', 0)
            
            # Calculate network latency
            network_latency = total_time - generation_time if 'generation_time' in response else 0
            
            run_result = {
                "input_text": input_text,
                "input_length": len(input_text.split()),
                "output_text": response.get('generated_text', ''),
                "output_length": len(response.get('generated_text', '').split()),
                "num_tokens": response.get('num_tokens', 0),
                "total_time": total_time,
                "generation_time": generation_time,
                "network_latency": network_latency,
                "tokens_per_second": response.get('num_tokens', 0) / generation_time if 'generation_time' in response and response.get('generation_time', 0) > 0 else 0,
                "error": response.get('error', None)
            }
            
            run_results.append(run_result)
        
        # Calculate average performance
        avg_result = {
            "input_text": input_text,
            "input_length": run_results[0]["input_length"],
            "output_text": run_results[0]["output_text"],
            "output_length": run_results[0]["output_length"],
            "num_tokens": run_results[0]["num_tokens"],
            "total_time": np.mean([r["total_time"] for r in run_results]),
            "generation_time": np.mean([r["generation_time"] for r in run_results]),
            "network_latency": np.mean([r["network_latency"] for r in run_results]),
            "tokens_per_second": np.mean([r["tokens_per_second"] for r in run_results]),
            "error": run_results[0]["error"]
        }
        
        results.append(avg_result)
    
    return results

# Define benchmark inputs
benchmark_inputs = [
    "def fibonacci(n):",
    "Write a function to calculate the factorial of a number.",
    "Explain the concept of recursion in programming.",
    "class BinarySearchTree:",
    "def quicksort(arr):"
]

# Run benchmark
benchmark_results = benchmark_api(
    inputs=benchmark_inputs,
    max_new_tokens=50,
    temperature=0.7,
    host=host,
    port=port,
    n_runs=3
)

# Convert to DataFrame for analysis
results_df = pd.DataFrame(benchmark_results)
results_df

## 6. Visualize Performance Metrics

Let's visualize the performance metrics to better understand the API's behavior.

In [ ]:
def plot_performance_metrics(results_df):
    """Plot performance metrics."""
    plt.figure(figsize=(15, 10))
    
    # Plot 1: Generation Time vs. Input Length
    plt.subplot(2, 2, 1)
    plt.scatter(results_df['input_length'], results_df['generation_time'])
    plt.xlabel('Input Length (words)')
    plt.ylabel('Generation Time (seconds)')
    plt.title('Generation Time vs. Input Length')
    
    # Plot 2: Tokens per Second
    plt.subplot(2, 2, 2)
    plt.bar(range(len(results_df)), results_df['tokens_per_second'])
    plt.xlabel('Input Index')
    plt.ylabel('Tokens per Second')
    plt.title('Tokens per Second')
    plt.xticks(range(len(results_df)), results_df['input_text'], rotation=45, ha='right')
    
    # Plot 3: Time Breakdown
    plt.subplot(2, 2, 3)
    ind = np.arange(len(results_df))
    width = 0.35
    plt.bar(ind, results_df['generation_time'], width, label='Generation Time')
    plt.bar(ind + width, results_df['network_latency'], width, label='Network Latency')
    plt.xlabel('Input Index')
    plt.ylabel('Time (seconds)')
    plt.title('Time Breakdown')
    plt.xticks(ind + width/2, range(len(results_df)))
    plt.legend()
    
    # Plot 4: Output Length vs. Generation Time
    plt.subplot(2, 2, 4)
    plt.scatter(results_df['num_tokens'], results_df['generation_time'])
    plt.xlabel('Output Length (tokens)')
    plt.ylabel('Generation Time (seconds)')
    plt.title('Output Length vs. Generation Time')
    
    plt.tight_layout()
    plt.savefig(output_dir / 'performance_metrics.png', dpi=300)
    plt.show()

# Plot performance metrics
plot_performance_metrics(results_df)

## 7. Test Batching and Caching

Let's test the batching and caching capabilities of the API by sending multiple requests simultaneously.

In [ ]:
def test_concurrent_requests(inputs, max_new_tokens=50, temperature=1.0, host='0.0.0.0', port=5000):
    """Test concurrent requests to the API."""
    results = []
    threads = []
    
    def make_request(input_text, results_list):
        start_time = time.time()
        response = generate_text(input_text, max_new_tokens, temperature, host, port)
        end_time = time.time()
        
        result = {
            "input_text": input_text,
            "output_text": response.get('generated_text', ''),
            "total_time": end_time - start_time,
            "generation_time": response.get('generation_time', 0),
            "error": response.get('error', None)
        }
        
        results_list.append(result)
    
    # Start threads for concurrent requests
    for input_text in inputs:
        thread = threading.Thread(target=make_request, args=(input_text, results))
        threads.append(thread)
        thread.start()
    
    # Wait for all threads to complete
    for thread in threads:
        thread.join()
    
    return results

# Test concurrent requests
concurrent_results = test_concurrent_requests(
    inputs=benchmark_inputs,
    max_new_tokens=50,
    temperature=0.7,
    host=host,
    port=port
)

# Convert to DataFrame
concurrent_df = pd.DataFrame(concurrent_results)
concurrent_df

## 8. Test Caching Performance

Let's test the caching performance by sending the same requests multiple times.

In [ ]:
def test_caching_performance(inputs, max_new_tokens=50, temperature=1.0, host='0.0.0.0', port=5000, n_repeats=3):
    """Test caching performance by repeating the same requests."""
    all_results = []
    
    for i in range(n_repeats):
        logger.info(f"Run {i+1}/{n_repeats}")
        
        results = []
        for input_text in inputs:
            start_time = time.time()
            response = generate_text(input_text, max_new_tokens, temperature, host, port)
            end_time = time.time()
            
            result = {
                "input_text": input_text,
                "run": i+1,
                "total_time": end_time - start_time,
                "generation_time": response.get('generation_time', 0),
                "error": response.get('error', None)
            }
            
            results.append(result)
        
        all_results.extend(results)
    
    return all_results

# Test caching performance
caching_results = test_caching_performance(
    inputs=benchmark_inputs[:2],  # Use a subset of inputs
    max_new_tokens=50,
    temperature=0.7,
    host=host,
    port=port,
    n_repeats=3
)

# Convert to DataFrame
caching_df = pd.DataFrame(caching_results)

# Pivot the DataFrame for easier visualization
pivot_df = caching_df.pivot(index='input_text', columns='run', values='total_time')
pivot_df.columns = [f'Run {i}' for i in pivot_df.columns]
pivot_df

## 9. Visualize Caching Performance

Let's visualize the caching performance to see how response times improve with caching.

In [ ]:
def plot_caching_performance(caching_df):
    """Plot caching performance."""
    plt.figure(figsize=(12, 6))
    
    # Group by input_text and run
    grouped = caching_df.groupby(['input_text', 'run'])['total_time'].mean().reset_index()
    
    # Plot for each input
    for input_text in grouped['input_text'].unique():
        input_data = grouped[grouped['input_text'] == input_text]
        plt.plot(input_data['run'], input_data['total_time'], marker='o', label=input_text)
    
    plt.xlabel('Run Number')
    plt.ylabel('Total Time (seconds)')
    plt.title('Caching Performance: Response Time vs. Run Number')
    plt.legend()
    plt.grid(True)
    plt.savefig(output_dir / 'caching_performance.png', dpi=300)
    plt.show()

# Plot caching performance
plot_caching_performance(caching_df)

## 10. Generate Deployment Documentation

Let's generate documentation for deploying the Flask API in production environments.

In [ ]:
def generate_deployment_docs():
    """Generate deployment documentation."""
    docs = """
# Flask API Deployment Guide

This guide provides instructions for deploying the Flask API for the AI model in production environments.

## Prerequisites

- Python 3.8 or higher
- PyTorch 1.13 or higher
- Flask 2.0 or higher
- CUDA-compatible GPU (recommended for production)

## Installation

1. Clone the repository:
```bash
git clone https://github.com/yourusername/advanced-ai-model-workflow.git
cd advanced-ai-model-workflow
```

2. Install dependencies:
```bash
pip install -r requirements.txt
```

## Configuration

1. Prepare the model and tokenizer:
   - Ensure the trained model checkpoint is available at `outputs/checkpoints/model_final.pt`
   - Ensure the tokenizer is available at `outputs/tokenizer/tokenizer.json`
   - Convert the YAML configuration to JSON using the provided script

2. Configure the server:
   - Edit the `config.yaml` file to adjust Flask deployment settings
   - Key settings include:
     - `host`: Server host (default: 0.0.0.0)
     - `port`: Server port (default: 5000)
     - `batch_size`: Maximum batch size for inference (default: 8)
     - `max_cache_size`: Maximum number of items to cache (default: 1000)

## Deployment Options

### Option 1: Direct Deployment

Run the Flask server directly:

```bash
python src/deployment/flask_server.py \
  --model-path outputs/checkpoints/model_final.pt \
  --tokenizer-path outputs/tokenizer/tokenizer.json \
  --config-path outputs/flask_deployment/config.json \
  --host 0.0.0.0 \
  --port 5000 \
  --batch-size 8 \
  --max-cache-size 1000 \
  --max-sequence-length 1024
```

### Option 2: Production Deployment with Gunicorn

For production environments, use Gunicorn as a WSGI server:

1. Create a WSGI entry point (`wsgi.py`):
```python
from src.deployment.flask_server import app, model_server
import os
import json

# Initialize model server
model_server = ModelServer(
    model_path=os.environ.get('MODEL_PATH', 'outputs/checkpoints/model_final.pt'),
    tokenizer_path=os.environ.get('TOKENIZER_PATH', 'outputs/tokenizer/tokenizer.json'),
    config_path=os.environ.get('CONFIG_PATH', 'outputs/flask_deployment/config.json'),
    device=os.environ.get('DEVICE', 'cuda' if torch.cuda.is_available() else 'cpu'),
    batch_size=int(os.environ.get('BATCH_SIZE', 8)),
    max_cache_size=int(os.environ.get('MAX_CACHE_SIZE', 1000)),
    max_sequence_length=int(os.environ.get('MAX_SEQUENCE_LENGTH', 1024))
)

if __name__ == '__main__':
    app.run()
```

2. Run with Gunicorn:
```bash
gunicorn --bind 0.0.0.0:5000 --workers 1 --threads 4 wsgi:app
```

### Option 3: Docker Deployment

1. Create a Dockerfile:
```dockerfile
FROM python:3.8-slim

WORKDIR /app

# Install dependencies
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Copy application code
COPY . .

# Set environment variables
ENV MODEL_PATH=/app/outputs/checkpoints/model_final.pt
ENV TOKENIZER_PATH=/app/outputs/tokenizer/tokenizer.json
ENV CONFIG_PATH=/app/outputs/flask_deployment/config.json
ENV DEVICE=cuda
ENV BATCH_SIZE=8
ENV MAX_CACHE_SIZE=1000
ENV MAX_SEQUENCE_LENGTH=1024

# Expose port
EXPOSE 5000

# Run the application
CMD ["gunicorn", "--bind", "0.0.0.0:5000", "--workers", "1", "--threads", "4", "wsgi:app"]
```

2. Build and run the Docker container:
```bash
docker build -t ai-model-api .
docker run -p 5000:5000 --gpus all ai-model-api
```

## API Usage

### Generate Text

**Endpoint:** `/generate`

**Method:** POST

**Request Body:**
```json
{
  "text": "def fibonacci(n):",
  "max_new_tokens": 50,
  "temperature": 0.7
}
```

**Response:**
```json
{
  "generated_text": "    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    else:\n        return fibonacci(n-1) + fibonacci(n-2)",
  "input_text": "def fibonacci(n):",
  "generation_time": 0.1234,
  "num_tokens": 42
}
```

### Health Check

**Endpoint:** `/health`

**Method:** GET

**Response:**
```json
{
  "status": "healthy"
}
```

## Performance Considerations

- **Batching:** The server automatically batches requests for improved throughput
- **Caching:** Responses are cached to improve performance for repeated requests
- **GPU Memory:** Monitor GPU memory usage and adjust batch size accordingly
- **Scaling:** For high-traffic deployments, consider load balancing across multiple instances

## Monitoring and Logging

- Logs are written to `flask_server.log`
- Consider integrating with monitoring tools like Prometheus and Grafana
- Key metrics to monitor:
  - Request latency
  - Throughput (requests per second)
  - GPU memory usage
  - Cache hit rate

## Troubleshooting

- **Out of Memory Errors:** Reduce batch size or max sequence length
- **Slow Response Times:** Check GPU utilization and consider upgrading hardware
- **Connection Errors:** Verify network configuration and firewall settings
"""
    
    # Save documentation
    docs_path = output_dir / 'deployment_guide.md'
    with open(docs_path, 'w') as f:
        f.write(docs)
    
    logger.info(f"Deployment documentation saved to {docs_path}")
    
    return docs

# Generate deployment documentation
deployment_docs = generate_deployment_docs()
print("Deployment documentation generated successfully!")

## 11. Clean Up

Finally, let's clean up by stopping the Flask server.

In [ ]:
# Stop the Flask server
if server_process is not None:
    logger.info("Stopping Flask server...")
    server_process.terminate()
    server_process.wait()
    logger.info("Flask server stopped")
else:
    logger.warning("Flask server was not running")

## 12. Summary

In this notebook, we've demonstrated the complete process of deploying the trained AI model as a Flask web service. The key steps were:

1. Setting up a Flask server with proper API endpoints
2. Implementing efficient batching and caching for improved performance
3. Testing the deployed model with various inputs
4. Measuring performance metrics like latency and throughput
5. Generating deployment documentation for production environments

The Flask API is now ready for deployment in production environments, with robust error handling, efficient batching and caching, and comprehensive documentation.